In [14]:
import pandas as pd
import geopandas as gpd
import sys
import os
import numpy as np

In [15]:
input_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\land_use_prep"
output_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\land_use_prep\processed"

## inputs

In [16]:
block_to_taz_file = os.path.join(input_dir, "zonal", "shp", "taz_2010block_allocation_20230314.csv")

# zone shape file
zone_shape_file = os.path.join(input_dir, "zonal", "shp", "CTPS_TDM23_TAZ_2017g_v202303.shp")

# emp input files
ma_emp_input_file = os.path.join(input_dir, "zonal", "ma_employment_run_113-209_2019_v20250829.csv")
nhri_emp_input_file = os.path.join(input_dir, "zonal", "nhri_employment_2020_v20230518.csv")

# pop input files
ma_pop_input_file = os.path.join(input_dir, "zonal", "ma_population_run_113-209_2019_v20250829.csv")
nhri_pop_input_file = os.path.join(input_dir, "zonal", "nhri_population_2020_v20230518.csv")

# enrollment input files
enroll_file = os.path.join(input_dir, "zonal", "enroll_v20240709.csv")

# parking
parking_file = os.path.join(input_dir, "zonal", "parking_v20221007.csv")

# terminal times
terminal_time_file = os.path.join(input_dir, "zonal", "terminal_times_v20221118.csv")
# transit access density
access_density_file = os.path.join(input_dir, "zonal", "access_density.csv")

# county FIPS
ma_county_fips_file = os.path.join(input_dir, "tl_2025_25_cousub", "tl_2025_25_cousub.shp")
nh_county_fips_file = os.path.join(input_dir, "tl_2020_33_cousub", "tl_2020_33_cousub.shp")
ri_county_fips_file = os.path.join(input_dir, "tl_2022_44_cousub", "tl_2022_44_cousub.shp")

In [17]:
block_to_taz_df = pd.read_csv(block_to_taz_file)

zone_shape_df = gpd.read_file(zone_shape_file)

ma_emp_df = pd.read_csv(ma_emp_input_file)
nhri_emp_df = pd.read_csv(nhri_emp_input_file)

ma_pop_df = pd.read_csv(ma_pop_input_file)
nhri_pop_df = pd.read_csv(nhri_pop_input_file)

enroll_df = pd.read_csv(enroll_file)
parking_df = pd.read_csv(parking_file)
terminal_time_df = pd.read_csv(terminal_time_file)
access_density_df = pd.read_csv(access_density_file)

ma_county_fips_df = gpd.read_file(ma_county_fips_file)
nh_county_fips_df = gpd.read_file(nh_county_fips_file)
ri_county_fips_df = gpd.read_file(ri_county_fips_file)



prepare land use attributs 
https://github.com/CTPSSTAFF/lighthouse/blob/e8479d64a4e50506fc9fdd42890e89e8446a59be/model/configs/settings.yaml#L58-L90

## zonal data

Assumptions
- COUNTY: county FIPS, externals use 99
- DISTRICT: district from shp
- SD: state from shp
- TOTACRE: total_area from shp, convert to acre
- RESACRE: TOTACRE*0.5
- CIACRE: TOTACRE*0.1
- area_type: urban from shp
- TOPOLOGY: 1 for all
- TERMINAL: 4 for urban, 1 or others

In [18]:
county_gdfs = []

for state_gdf, state_name in [(ma_county_fips_df, "MA"), (nh_county_fips_df, "NH"), (ri_county_fips_df, "RI")]:
    if state_gdf is not None:
        county_gdfs.append(state_gdf)

In [19]:
combined_counties = pd.concat(county_gdfs, ignore_index=True)

In [20]:
print(combined_counties.crs)
print(zone_shape_df.crs)

EPSG:4269
EPSG:26986


In [21]:
combined_counties = combined_counties.to_crs(zone_shape_df.crs)

In [22]:
taz_centroids = zone_shape_df.copy()
taz_centroids["geometry"] = taz_centroids.geometry.centroid

result = gpd.sjoin(
    taz_centroids, 
    combined_counties[[ "geometry", "COUNTYFP" ]], 
    how="left", 
    predicate="within"
)
result_grouped = result.groupby(result.index).first()

# county id
zone_shape_df["COUNTY"] = result_grouped["COUNTYFP"].astype('Int64')
zone_shape_df.loc[zone_shape_df["type"] == 'E', "COUNTY"] = 99

# square miles to acres
# TODO: update RESACRE and CIACRE with actual land use data
zone_shape_df["TOTACRE"] = zone_shape_df["total_area"] * 640
zone_shape_df["RESACRE"] = zone_shape_df["TOTACRE"] * 0.5
zone_shape_df["CIACRE"] = zone_shape_df["TOTACRE"] * 0.1

# superdistricts
zone_shape_df["SD"] = 99
zone_shape_df.loc[zone_shape_df['state']=="MA", 'SD'] = 25
zone_shape_df.loc[zone_shape_df['state']=="NH", 'SD'] = 33
zone_shape_df.loc[zone_shape_df['state']=="RI", 'SD'] = 44

zone_shape_df.rename(columns={"district": "DISTRICT"}, inplace=True)
zone_shape_df.rename(columns={"taz_id": "TAZ"}, inplace=True)
zone_shape_df["TOPOLOGY"] = 1

In [23]:
# area type and terminal times
access_density_df = access_density_df.merge(
    terminal_time_df[["access_density","terminal_time_p"]],
    on="access_density",
    how="left"
)

access_density_df.rename(
    columns={"taz_id": "TAZ", "access_density": "area_type", "terminal_time_p": "TERMINAL"}, 
    inplace=True
)

In [24]:
zone_shape_df = zone_shape_df.merge(
    access_density_df[["TAZ", "area_type", "TERMINAL"]],
    on="TAZ",
    how="left"
)

In [25]:
zone_shape_df = zone_shape_df[["TAZ", "COUNTY", "DISTRICT", "SD", "TOTACRE", "RESACRE", "CIACRE", "area_type", "TERMINAL", "TOPOLOGY"]]
zone_shape_df = zone_shape_df.sort_values("TAZ").reset_index(drop=True)

In [26]:
zone_shape_df

,TAZ,COUNTY,DISTRICT,SD,TOTACRE,RESACRE,CIACRE,area_type,TERMINAL,TOPOLOGY
0,1,25,0,25,45.44,22.72,4.544,1,5,1
1,2,25,0,25,14.08,7.04,1.408,1,5,1
2,3,25,0,25,19.84,9.92,1.984,1,5,1
3,4,25,0,25,17.92,8.96,1.792,1,5,1
4,5,25,0,25,27.52,13.76,2.752,2,4,1
...,...,...,...,...,...,...,...,...,...,...
5834,209096,99,89,99,0.00,0.00,0.000,6,1,1
5835,209097,99,89,99,0.00,0.00,0.000,6,1,1
5836,209098,99,89,99,0.00,0.00,0.000,6,1,1
5837,209099,99,89,99,0.00,0.00,0.000,6,1,1


## employment

In [27]:
def process_employment_data(employment_df, block_to_taz_df):
    merged = employment_df.merge(block_to_taz_df, on='block_id', how='left')

    employment_columns = [col for col in merged.columns 
                        if col not in ['block_id', 'taz_id', 'area_fct']]

    # apply area factor
    for col in employment_columns:
        merged[col] = merged[col] * merged['area_fct']

    taz_employment = merged.groupby('taz_id')[employment_columns].sum().reset_index()

    taz_employment['RETEMPN'] = taz_employment['6_ret_leis']
    taz_employment['FPSEMPN'] = taz_employment['3_finance'] + taz_employment['9_profbus']
    taz_employment['HEREMPN'] = taz_employment['2_eduhlth']
    taz_employment['OTHEMPN'] = taz_employment['1_constr'] + taz_employment['4_public'] + taz_employment['5_info'] + taz_employment['8_other']
    taz_employment['AGREMPN'] = 0 
    taz_employment['MWTEMPN'] = taz_employment['7_manu'] + taz_employment['10_ttu']

    taz_employment = taz_employment[['taz_id', 'total_households', 'total_jobs', 'RETEMPN', 'FPSEMPN', 'HEREMPN', 'OTHEMPN', 'AGREMPN', 'MWTEMPN']]
    taz_employment = taz_employment.rename(
        columns={'total_households': 'TOTHH', 
                 'total_jobs': 'TOTEMP',
                 'taz_id': 'TAZ'}
    )

    taz_employment = taz_employment.sort_values('TAZ').reset_index(drop=True)

    return taz_employment

In [28]:
ma_emp_agg = process_employment_data(ma_emp_df, block_to_taz_df)
nhri_emp_agg = process_employment_data(nhri_emp_df, block_to_taz_df)

emp_combined = pd.concat([ma_emp_agg, nhri_emp_agg], ignore_index=False)
emp_combined = emp_combined.set_index('TAZ')
emp_combined = emp_combined.groupby(level=0).sum().reset_index()
emp_combined = emp_combined.sort_values('TAZ').reset_index(drop=True)

In [29]:
zone_shape_df = zone_shape_df.merge(emp_combined, on='TAZ', how='left').fillna(0)

## population

In [30]:
def process_population_data(population_df, block_to_taz_df):
    merged = population_df.merge(block_to_taz_df, on='block_id', how='left')

    merged['is_age_0519'] = ((merged['age'] >= 5) & (merged['age'] <= 19)).astype(int)

    # apply area factor
    merged['pop_weighted'] = merged['area_fct']
    merged['pop_age0519_weighted'] = merged['area_fct'] * merged['is_age_0519']
    
    taz_population = merged.groupby('taz_id').agg(
        TOTPOP=('pop_weighted', 'sum'),
        AGE0519=('pop_age0519_weighted', 'sum')
    ).reset_index()
    
    taz_population = taz_population.rename(
        columns={'taz_id': 'TAZ'}
    )
    taz_population = taz_population.sort_values('TAZ').reset_index(drop=True)

    return taz_population

In [31]:
ma_pop_agg = process_population_data(ma_pop_df, block_to_taz_df)
nhri_pop_agg = process_population_data(nhri_pop_df, block_to_taz_df)

pop_combined = pd.concat([ma_pop_agg, nhri_pop_agg], ignore_index=False)
pop_combined = pop_combined.set_index('TAZ')
pop_combined = pop_combined.groupby(level=0).sum().reset_index()
pop_combined = pop_combined.sort_values('TAZ').reset_index(drop=True)

In [32]:
zone_shape_df = zone_shape_df.merge(pop_combined, on='TAZ', how='left').fillna(0)

## parking

In [33]:
zone_shape_df = zone_shape_df.merge(
    parking_df[['taz_id','cost_hr']].rename(columns={'taz_id': 'TAZ'}),
    on='TAZ', 
    how='left').fillna(0)

In [34]:
# TODO: update parking cost
zone_shape_df.rename(columns={'cost_hr': 'PRKCST'}, inplace=True)
zone_shape_df['OPRKCST'] = zone_shape_df['PRKCST']

## enrollment

In [35]:
enroll_df['HSENROLL'] = enroll_df['k12']

# TODO: update with actual enrollment data
enroll_df['COLLFTE'] = (enroll_df['college_total'] * 0.9).astype(int)
enroll_df['COLLPTE'] = (enroll_df['college_total'] * 0.1).astype(int)

In [36]:
zone_shape_df = zone_shape_df.merge(
    enroll_df[['taz_id','HSENROLL', 'COLLFTE', 'COLLPTE']].rename(columns={'taz_id': 'TAZ'}),
    on='TAZ', 
    how='left').fillna(0)

## output

In [37]:
zone_shape_df =  zone_shape_df[["TAZ", "COUNTY", "DISTRICT", "SD", 
                                "TOTHH", "TOTPOP", "TOTACRE", "RESACRE", "CIACRE", "TOTEMP", "AGE0519",
                                "RETEMPN", "FPSEMPN", "HEREMPN", "OTHEMPN", "AGREMPN", "MWTEMPN",
                                "PRKCST", "OPRKCST", 
                                "area_type", "HSENROLL", "COLLFTE", "COLLPTE",
                                "TERMINAL", "TOPOLOGY"
                                ]].to_csv(os.path.join(output_dir, "land_use.csv"), index=False)